# How does access to early retinal fates change across embryogenesis?

Retinal ganglion cells are produced early in retinogenesis, while production of other neurons overlaps and changes over time. Clark and colleagues distinguished primary retinal progenitors from a neurogenic compartment and described changes in developmental competence. We start from their **Neurogenic Cells** at E12, E14, E16 and E18. We ask whether cells carrying that same annotation differ in their modeled access to RGC, amacrine and cone states. These are three observed destinations within a larger retinal lineage. A common immediate three-way division is not assumed. The NFI manipulation in the publication is a separate experiment. [Clark et al., PMID 31128945](https://pubmed.ncbi.nlm.nih.gov/31128945/)

Begin with the six looms from the accompanying preparation guide. This notebook is designed for local execution. Every directional result below is computed from the spliced and unspliced layers you supply.

## What was sampled?

Use the original authors' annotation, including their horizontal-cell update. Examine each library before comparing ages. A missing local endpoint matters even when that type is well represented elsewhere in the atlas.

In [1]:
from pathlib import Path
import os
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import scCS
import matplotlib.pyplot as plt
from scipy import sparse
from IPython.display import display

cwd = Path.cwd()
candidates = [cwd, *(p / "tutorials/resources/clark_retina" for p in (cwd, *cwd.parents))]
SRC = next(p for p in candidates if (p / "cells.tsv.gz").is_file())
WORK = Path(os.environ.get("RETINA_WORK", Path.home() / "projects/clark-retina")).expanduser()
OUT = WORK / "results"
OUT.mkdir(exist_ok=True, parents=True)
samples = pd.read_csv(SRC / "samples.tsv", sep="\t")
meta = pd.read_csv(SRC / "cells.tsv.gz", sep="\t")
AGES = ["E12", "E14", "E16", "E18"]
ROOT = "Neurogenic Cells"
FATES = ["Retinal Ganglion Cells", "Amacrine Cells", "Cones"]
assert not meta.duplicated(["library", "barcode"]).any()
design = pd.crosstab(meta.library, meta.author_celltype).reindex(samples.library)
display(design[[ROOT, *FATES]])
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none"})


def save(fig, stem):
    for axis in fig.axes:
        for artist in axis.collections:
            artist.set_rasterized(False)
    fig.savefig(OUT / f"{stem}.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)

author_celltype,Neurogenic Cells,Retinal Ganglion Cells,Amacrine Cells,Cones
library,,,,
E12,25,8,1,0
E14_rep1,1519,2742,1,15
E14_rep2,1218,1776,54,153
E16,510,954,11,37
E18_rep2,1091,192,1731,612
E18_rep3,1204,309,1939,660


## Join measured RNA layers to the published labels

Match by library and 10x barcode. Preserve Ensembl IDs while combining looms. Record all unmatched cells so that a reduction in one of the rarer fates can be traced back to its library. The layers contain molecule counts. The native velocity fit will retain the broader set of retinal identities as graph context.

In [ ]:
parts, audit, missing_rows = [], [], []
for lib in samples.library:
    loom = WORK / "looms" / f"{lib}.loom"
    x = ad.read_loom(loom, obs_names="CellID", var_names="Accession", X_name="spliced")
    assert x.var_names.is_unique and {"spliced", "unspliced"}.issubset(x.layers)
    for layer in ("spliced", "unspliced"):
        mat = sparse.csr_matrix(x.layers[layer])
        assert mat.nnz and np.all(np.isfinite(mat.data)) and mat.data.min() >= 0
    barcode = x.obs_names.to_series().str.extract(r"([ACGT]{16})(?:-1|x)?$", expand=False)
    if barcode.isna().any() or barcode.duplicated().any():
        raise ValueError(f"Unrecognized or repeated barcode in {lib}.")
    x.obs_names = pd.Index(barcode.to_numpy() + "-1")
    ref = meta[meta.library.eq(lib)].set_index("barcode")
    matched = ref.index[ref.index.isin(x.obs_names)]
    if not len(matched):
        raise ValueError(f"No annotation matches in {lib}. Check library labels and CellID.")
    for key in ref.index.difference(matched):
        missing_rows.append((lib, key, ref.at[key, "author_celltype"], "annotation_only"))
    for key in x.obs_names.difference(matched):
        missing_rows.append((lib, key, None, "loom_only"))
    audit.append((lib, len(ref), x.n_obs, len(matched)))
    x = x[matched].copy()
    x.obs = ref.loc[matched].copy()
    x.obs["barcode"] = matched
    x.obs_names = pd.Index([f"{lib}:{v}" for v in matched])
    parts.append(x)

tab = pd.DataFrame(audit, columns=["library", "annotated", "loom_cells", "matched"])
display(tab)
tab.to_csv(OUT / "cell_matching.csv", index=False)
pd.DataFrame(missing_rows, columns=["library", "barcode", "celltype", "source"]).to_csv(
    OUT / "unmatched_cells.csv", index=False
)
symbols = parts[0].var["Gene"].copy()
a = ad.concat(parts, join="inner", merge="first")
del parts
a.var["ensembl_id"] = a.var_names
a.var_names = pd.Index(symbols.reindex(a.var_names).astype(str))
a.var_names_make_unique()
a.obs["age"] = pd.Categorical(a.obs.age, categories=AGES, ordered=True)
a.obs["author_celltype"] = a.obs.author_celltype.astype("category")
display(pd.crosstab(a.obs.library, a.obs.author_celltype)[[ROOT, *FATES]])

## Fit retinal RNA kinetics and inspect the native field

Fit the reference across ages on the annotated retinal cells. Ages and outcome labels are withheld from gene fitting and from construction of the nearest-neighbor graph. Inspect the relative positions of progenitors and differentiating neurons, RNA-velocity confidence, and spliced versus unspliced expression for genes with fitted kinetics. The pooled reference lets a progenitor connect to a state observed later in development.

In [ ]:
np.random.seed(19)
scv.pp.filter_genes(a, min_shared_counts=20)
scv.pp.normalize_per_cell(a)
sc.pp.log1p(a)
sc.pp.highly_variable_genes(a, n_top_genes=2000)
markers = a.var_names.isin(
    ["Atoh7", "Neurog2", "Pou4f2", "Isl1", "Tfap2a", "Tfap2b", "Otx2", "Thrb", "Rxrg"]
)
a = a[:, a.var.highly_variable.to_numpy() | markers].copy()
sc.pp.pca(a, n_comps=30, random_state=19)
sc.pp.neighbors(a, n_neighbors=30, n_pcs=30, random_state=19)
sc.tl.umap(a, random_state=19)
scv.pp.moments(a, n_neighbors=None, n_pcs=None)
scv.tl.recover_dynamics(a, n_top_genes=150, max_iter=20, n_jobs=1, show_progress_bar=False)
scv.tl.velocity(a, mode="dynamical")
scv.tl.velocity_graph(a, n_jobs=1, show_progress_bar=False)
scv.tl.velocity_confidence(a)
scv.tl.velocity_pseudotime(a)
T = scCS.get_scvelo_transition_matrix(a).tocsr()
assert T.shape == (a.n_obs, a.n_obs) and np.isfinite(T.data).all() and T.data.min() >= 0
np.testing.assert_allclose(np.asarray(T.sum(axis=1)).ravel(), 1, atol=1e-6)
assert (T - sparse.diags(T.diagonal())).nnz
a.obsp["T_scCS_velocity"] = T
a.write_h5ad(OUT / "retina_velocity.h5ad", compression="gzip")
display(a.obs.groupby("library", observed=True).velocity_confidence.agg(["count", "median"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
scv.tl.velocity_embedding(a, basis="umap")
sc.pl.umap(a, color="author_celltype", ax=axes[0], show=False)
xy, uv = a.obsm["X_umap"], a.obsm["velocity_umap"]
speed = np.linalg.norm(uv, axis=1)
valid = np.flatnonzero(np.isfinite(uv).all(axis=1) & (speed > 0))
if not len(valid):
    raise ValueError("The fitted kinetic field has no finite, nonzero projected velocity.")
display_ids = np.random.default_rng(19).choice(valid, min(1000, len(valid)), replace=False)
scale = np.quantile(speed[valid], 0.90) / (0.04 * np.ptp(xy, axis=0).max())
axes[0].quiver(
    *xy[display_ids].T,
    *uv[display_ids].T,
    angles="xy",
    scale_units="xy",
    scale=scale,
    width=0.002,
    color="#253747",
    alpha=0.65,
)
axes[0].set_title("Retinal RNA velocity")
sc.pl.umap(a, color="age", ax=axes[1], show=False)
fig.tight_layout()
save(fig, "native_retina_velocity")
genes = a.var.loc[a.var.velocity_genes, "fit_likelihood"].dropna().nlargest(3).index
if not len(genes):
    raise ValueError("No kinetic genes passed the fit criteria.")
fig, axes = plt.subplots(1, len(genes), figsize=(5 * len(genes), 4), squeeze=False)
for gene, ax in zip(genes, axes[0]):
    scv.pl.scatter(a, basis=gene, color="author_celltype", legend_loc="none", ax=ax, show=False)
fig.tight_layout()
save(fig, "kinetic_phase_portraits")
present = [g for g in ["Atoh7", "Pou4f2", "Tfap2b", "Rxrg"] if g in a.var_names]
if present:
    sc.pl.umap(a, color=present, ncols=2, show=False)
    save(plt.gcf(), "retinal_identity_markers")

## Which destinations remain accessible to neurogenic progenitors?

Count first entry into any annotated RGC, amacrine or cone cell. This definition of arrival can capture an early postmitotic identity. The three populations differ in abundance and in the maturity of the cells they contain. Keep the same ordering, endpoints and 64-step horizon in the pooled and age-restricted models. The second model asks what each age can resolve using only its own transitions.

In [ ]:
def fit_model(mode, scope):
    model = scCS.MultiScorer(
        a,
        root=ROOT,
        branches=FATES,
        obs_key="author_celltype",
        condition_obs_key="age",
        replicate_obs_key="library",
        condition_order=AGES,
    )
    diagnostics = model.preflight(ordering_metric="velocity_pseudotime", check_velocity=True)
    # E12 and E16 each have one library. Both are used descriptively.
    singleton_codes = {"condition_E12_replicates", "condition_E16_replicates"}
    unexpected = [d for d in diagnostics.errors if (d.code not in singleton_codes or d.value != 1)]
    if unexpected:
        raise ValueError("Preflight: " + " | ".join(d.message for d in unexpected))
    if mode == "future_fate" and scope == "pooled":
        display(diagnostics.to_frame())
    model.build_embedding(ordering_metric="velocity_pseudotime", cache_subset=False, verbose=False)
    options = (
        {"future_fate_options": {"effective_horizon": 64, "min_anchor_cells": a.n_obs}}
        if mode == "future_fate"
        else {}
    )
    model.fit(
        scoring_mode=mode, transition_scope=scope, transition_matrix=T, verbose=False, **options
    )
    result = model.score_all_conditions(
        population="root", min_cells=1, min_replicates=1, write_to_adata=False, verbose=False
    )
    return model, result


multi, pooled = fit_model("future_fate", "pooled")
_, within = fit_model("future_fate", "condition")
rows = []
for scope, results in [("pooled", pooled), ("within_age", within)]:
    for age, r in results.items():
        ids = pd.Index(r.cell_ids)
        libraries = a.obs.loc[ids, "library"]
        for lib in libraries.unique():
            mask = libraries.eq(lib).to_numpy()
            defined = mask & np.asarray(r.affinity_defined)
            for j, fate in enumerate(r.fate_names):
                rows.append(
                    [
                        scope,
                        age,
                        lib,
                        fate,
                        int(mask.sum()),
                        int(defined.sum()),
                        r.commitment_strength[mask].mean(),
                        r.commitment_contribution[mask, j].mean(),
                        r.directional_affinity[defined, j].mean() if defined.any() else np.nan,
                    ]
                )
scores = pd.DataFrame(
    rows,
    columns=[
        "scope",
        "age",
        "library",
        "destination",
        "roots",
        "defined",
        "reach",
        "contribution",
        "affinity",
    ],
)
display(scores.round(3))
scores.to_csv(OUT / "library_fate_scores.csv", index=False)
metrics = ["reach", "contribution", "affinity"]
age_means = scores.groupby(["scope", "age", "destination"], observed=True)[metrics].agg(
    lambda col: col.mean(skipna=False)
)
age_means.to_csv(OUT / "age_means.csv")
for scope in ("pooled", "within_age"):
    table = age_means.loc[scope]
    delta = table.loc["E18"] - table.loc["E12"]
    display(delta.round(3))
    delta.to_csv(OUT / f"E18_minus_E12_{scope}.csv")

## Read age differences alongside their denominator

Look first at RGC absolute contribution, then at total reach and the number of progenitors with defined affinity. A change in conditional preference has a different meaning if most cells acquire or lose resolvable endpoints. Points at E14 and E18 allow inspection of agreement between libraries. At E12 and E16 there is one library per age; the age contrasts are descriptive. An early root can reach a cone on the pooled reference even though the E12 annotation contains no cones.

In [ ]:
fig, axes = plt.subplots(1, len(FATES), figsize=(14, 4), sharey=True)
for ax, fate in zip(axes, FATES):
    for scope, style in [("pooled", "o-"), ("within_age", "s--")]:
        subset = scores[scores.scope.eq(scope) & scores.destination.eq(fate)]
        vals = subset.groupby("age", observed=True).contribution.mean().reindex(AGES)
        ax.plot(range(len(AGES)), vals, style, label=scope)
        for i, age in enumerate(AGES):
            yy = subset.loc[subset.age.eq(age), "contribution"].to_numpy()
            off = np.linspace(-0.06, 0.06, len(yy)) if len(yy) > 1 else [0.0]
            ax.scatter(i + np.array(off), yy, s=22, facecolors="none", edgecolors="black")
    ax.set_xticks(range(len(AGES)), AGES)
    ax.set_title(fate)
axes[0].set_ylabel("Mean absolute contribution")
axes[-1].legend()
fig.tight_layout()
save(fig, "age_and_library_contributions")
for metric, name in [
    ("future_fate_reach", "reach"),
    ("future_fate_affinity:Retinal Ganglion Cells", "rgc_affinity"),
]:
    save(multi.plot_star_grid(pooled, color_by=metric, population="all", ncols=2), name + "_stars")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
multi.plot_commitment_heatmap(
    pooled, metric="future_fate_contribution", level="condition", ax=axes[0]
)
multi.plot_status_composition(pooled, ax=axes[1])
multi.plot_transition_coverage(pooled, ax=axes[2])
fig.tight_layout()
save(fig, "fate_composition_and_coverage")

## Does the immediate velocity agree with access to a later state?

The rose grids describe the one-step directions of neurogenic cells at each age in the supervised fate geometry. The preceding contribution plots follow longer paths. A mismatch can identify an intermediate state worth examining in the native field or phase portraits. Neither quantity measures the fraction of progenitors whose daughters were observed to adopt a given fate.

In [ ]:
local, local_results = fit_model("instantaneous", "pooled")
rose = local.plot_rose_grid(
    local_results,
    population="root",
    mode="branch",
    n_bins=30,
    ncols=2,
    title="Fate-directed velocity of neurogenic retinal cells",
)
save(rose, "neurogenic_velocity_roses")